# HGRIA - Hand Gesture Recognition for Interactive Applications
## Local Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE (LOCAL)                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Local Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│                                                 │           │
│                                            <project>/logs/  │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Python 3.10+ with pip
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)
- All dependencies installed (`pip install -r requirements.txt`)

### How it works
1. Backend runs Flask server locally with MediaPipe
2. ngrok creates HTTPS tunnel to expose backend
3. Frontend connects via WebSocket and sends webcam frames
4. Backend processes frames and sends gesture commands back

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Step 1: Verify dependencies
import numpy as np
import google.protobuf
import mediapipe as mp
import tensorflow as tf

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe OK")
print("TensorFlow OK")

In [ ]:
# Step 2: Set up project root and Python path
import os
import sys
from pathlib import Path

# Notebook lives at <project_root>/notebooks/ — walk up one level
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

print(f"✓ Project root : {PROJECT_ROOT}")

In [ ]:
# Step 3: Install pyngrok and configure ngrok authentication
#
# ── EDIT THIS ──────────────────────────────────────────────────────────────
# Paste your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken
# Leave as empty string to use an anonymous tunnel (disconnects after ~2 h).
NGROK_AUTHTOKEN = "3Hwfa0sDIVcU5HO0Fxt46CaFAKB_3K8KDyZp8gt2PqPqytY9t"  # e.g. "2abc123XYZ_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# ───────────────────────────────────────────────────────────────────────────

import re
import subprocess

subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

token = NGROK_AUTHTOKEN.strip()
if token:
    # Basic sanity check: ngrok tokens are alphanumeric/underscore/dash, 20+ chars
    if not re.match(r'^[A-Za-z0-9_\-]{20,}$', token):
        raise ValueError(
            "NGROK_AUTHTOKEN does not look valid.\n"
            "Expected an alphanumeric string of 20+ characters.\n"
            "Get yours from: https://dashboard.ngrok.com/get-started/your-authtoken"
        )
    ngrok.set_auth_token(token)
    print("\u2713 ngrok authenticated — tunnel will open when the server starts (Step 6)")
else:
    print("\u26a0 No authtoken set — anonymous tunnel (may disconnect after ~2 h)")
    print("  Paste your token into NGROK_AUTHTOKEN above to avoid this")

In [ ]:
# Step 4: Patch config for local environment
import json

config_path = str(PROJECT_ROOT / 'config' / 'config.json')

with open(config_path, 'r') as f:
    config = json.load(f)

# colab_mode=True: backend receives webcam frames POSTed from the browser
# via /api/frame instead of opening a local camera device.
# This is the correct mode for any setup where the browser supplies the
# camera (local notebook, Colab, or any machine without a free camera).
config['camera']['colab_mode'] = True
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True

log_dir = PROJECT_ROOT / 'logs'
log_dir.mkdir(exist_ok=True)
config['logging']['log_file_path'] = str(log_dir) + '/'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("\u2713 Config patched:")
print(f"  - colab_mode   : {config['camera']['colab_mode']}")
print(f"  - cors_origins : {config['server']['cors_origins']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

In [ ]:
# Step 5: Apply in-place patches and clear Python module cache
#
# This step does two things:
#
# 1. Patches configuration.py so it works even when the repo copy has not
#    been updated yet.  The patch fixes ConfigurationManager.__getattr__ to
#    return gesture_cooldowns_ms as a plain dict instead of wrapping it in
#    _Namespace (which has no .get()).
#
# 2. Evicts all backend.* modules from sys.modules so the next import always
#    reads the patched source rather than stale bytecode.
import os, sys

# ── Patch configuration.py ───────────────────────────────────────────────
_CFG_PATH = str(PROJECT_ROOT / 'backend' / 'core' / 'configuration.py')

with open(_CFG_PATH, 'r') as _f:
    _src = _f.read()

_OLD = '''    def __getattr__(self, name: str) -> Any:
        if name.startswith("_"):
            raise AttributeError(name)
        if name in self._data:
            val = self._data[name]
            if isinstance(val, dict):
                return _Namespace(val)
            return val
        raise AttributeError(f"No config section: {name}")'''

_NEW = '''    # Sections that must remain plain dicts (callers use dict methods on them).
    _PLAIN_DICT_SECTIONS = frozenset({"gesture_cooldowns_ms", "custom_gestures"})

    def __getattr__(self, name: str) -> Any:
        if name.startswith("_"):
            raise AttributeError(name)
        if name in self._data:
            val = self._data[name]
            if isinstance(val, dict) and name not in self._PLAIN_DICT_SECTIONS:
                return _Namespace(val)
            return val
        raise AttributeError(f"No config section: {name}")'''

if _OLD in _src:
    _src = _src.replace(_OLD, _NEW)
    with open(_CFG_PATH, 'w') as _f:
        _f.write(_src)
    print('\u2713 Patched configuration.py')
elif '_PLAIN_DICT_SECTIONS' in _src:
    print('\u2713 configuration.py already contains the fix — no patch needed')
else:
    print('\u26a0 Could not locate patch target in configuration.py — check manually')

# ── Evict cached modules ─────────────────────────────────────────────────
_PREFIXES = ('backend.', 'backend', 'dynamic_gestures.')
_evicted = [k for k in list(sys.modules) if k.startswith(_PREFIXES)]
for _mod in _evicted:
    del sys.modules[_mod]

print(f'\u2713 Evicted {len(_evicted)} cached module(s) — fresh import guaranteed')

In [ ]:
# Step 6: Start the HGRIA Server (blocking)
#
# After this cell runs you will see one of two messages:
#
#   WITH ngrok:
#     Public URL : https://xxxx.ngrok-free.app
#     Frontend   : https://qtannguyen-researcher.github.io/HGRIA/?server=https://xxxx.ngrok-free.app
#     → Open the Frontend URL in your browser.
#
#   WITHOUT ngrok (local only):
#     Server running at: http://localhost:5000
#     → Open http://localhost:5000 directly in your browser.
#       The frontend is served by Flask from the /frontend folder.
#
# Press the Stop button (■) or Kernel > Interrupt to terminate the server.
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print("-" * 60)

orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

## Post-Launch Instructions

### Accessing the Frontend

After Step 6 starts, look for the `Frontend :` line in the output and open that URL directly.
It looks like:
```
Frontend   : https://qtannguyen-researcher.github.io/HGRIA/?server=https://xxxx.ngrok-free.app
```

### If ngrok URL Changes

1. Stop the server (interrupt Step 6)
2. Re-run Steps 4 and 6
3. Use the new `Frontend :` URL printed in the output

### Troubleshooting

| Issue | Solution |
|-------|----------|
| ngrok URL changed | Re-run Steps 4 and 6, use new Frontend URL |
| `authtoken does not look valid` | Edit `NGROK_AUTHTOKEN` in Step 3 with a real token |
| Code changes not taking effect | Re-run Steps 4–6 |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| Port 5000 in use | Kill existing process: `lsof -ti:5000 \| xargs kill` |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm

In [ ]:
# Step 7 (Optional): Pipeline diagnostic — run in a NEW cell WHILE Step 6 is running
#
# Open http://localhost:5000/api/debug in your browser, or run this cell.
# It tells you exactly where in the pipeline frames are being dropped.
import urllib.request, json, time

for attempt in range(3):
    try:
        with urllib.request.urlopen('http://localhost:5000/api/debug', timeout=3) as r:
            data = json.loads(r.read())
        print(json.dumps(data, indent=2))
        print()
        print('>>> DIAGNOSIS:', data.get('diagnosis', 'n/a'))
        break
    except Exception as e:
        print(f'Attempt {attempt+1} failed: {e}')
        time.sleep(2)